In [1]:
from dataloader import *
from augmentations import *
from models import *
from training import *
import matplotlib.pyplot as plt
from itertools import product
from statistics import mean, stdev

## Experiment loop for hyperparameter selection

In [2]:
params = {
     #training params
    "batch_size": [8, 32, 128],
    "learning_rate": [1e-3, 1e-4, 1e-5],
    "optimizer": ['Adam', 'SGD'],
    #regularization params
    "dropout": [0.2, 0.5],
    "weight_decay": [0, 1e-3, 0.5]
}

# Create experiment grid
experiment_grid = [dict(zip(params.keys(), v)) for v in product(*params.values())]

In [3]:
# Only for testing!!!!!!
from torch.utils.data import Subset
import numpy as np

def get_subset(dataset, fraction=0.1):
    size = int(len(dataset) * fraction)
    indices = np.random.choice(len(dataset), size, replace=False)
    return Subset(dataset, indices)

In [4]:
def run_experiment(model_name,experiment_config, epoch_number=2):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data

    transformations = basic_transforms(augmentation_type=None, model_type=model_name)

    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    #!!! only for testing
    train_dataset = get_subset(train_dataset, fraction=0.005)
    val_dataset = get_subset(val_dataset, fraction=0.005)
    test_dataset = get_subset(test_dataset, fraction=0.005)
    
    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=None,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    accs = []
    f1s = []
    recalls = []
    precisions = []
    for epoch in range(epoch_number):

        train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device)

        accs.append(scores["accuracy"])
        f1s.append(scores["f1"])
        recalls.append(scores["recall"])
        precisions.append(scores["precision"])

    val_scores = {
        'mean_acc': mean(accs),
        'std_acc': stdev(accs),

        'mean_f1': mean(f1s),
        'std_f1': stdev(f1s),

        'mean_recall': mean(recalls),
        'std_recall': stdev(recalls),
        'mean_precision': mean(precisions),
        'std_precision': stdev(precisions)
    }

    test_scores = validate(model, test_loader, device)

    return {**val_scores, **test_scores}

In [ ]:
results = []

for i, config in enumerate(experiment_grid):

    if i ==3:
        break

    print(config)

    model = "ResNet" # do wyboru jeszcze ConvNeXt, VisionTransformer

    score = run_experiment(model,config)
    d = {
        "model": model,
        "config": config}
    results.append({
        **d, **score
    })

### Tutaj wybieramy najlepsze konfiguracje (jedna na kazdy model) i zaczynamy testowanie augmentacji

## Experiment loop for auqmentation techniques

In [10]:
# przykladowo
best_conf_resnet = {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}

In [11]:
# standard operations
basic_augmentations = ["flip", "shift", "rotation"]
# more advanced data augmentation
advanced_augmentations = 'cutmix' 

In [12]:
def run_experiment_augmentation(model_name,experiment_config,augmentation=None, cutmix_collate_fn=None, epoch_number=2):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data
    transformations = basic_transforms(augmentation_type=augmentation, model_type=model_name)


    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    #!!! only for testing
    train_dataset = get_subset(train_dataset, fraction=0.01)
    val_dataset = get_subset(val_dataset, fraction=0.01)
    test_dataset = get_subset(test_dataset, fraction=0.01)
    

    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=cutmix_collate_fn,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    accs = []
    f1s = []
    recalls = []
    precisions = []
    for epoch in range(epoch_number):

        train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device)

        accs.append(scores["accuracy"])
        f1s.append(scores["f1"])
        recalls.append(scores["recall"])
        precisions.append(scores["precision"])

    val_scores = {
        'mean_acc': mean(accs),
        'std_acc': stdev(accs),

        'mean_f1': mean(f1s),
        'std_f1': stdev(f1s),

        'mean_recall': mean(recalls),
        'std_recall': stdev(recalls),
        'mean_precision': mean(precisions),
        'std_precision': stdev(precisions)
    }

    test_scores = validate(model, test_loader, device)

    return {**val_scores, **test_scores}

In [ ]:
# augmentation experiments

results = []

for augm in basic_augmentations:

    model = "ResNet"

    score = run_experiment_augmentation(model,best_conf_resnet, augm, cutmix_collate_fn=None)
    d = {
        "model": model,
        "augmentation": augm}
    
    results.append({
        **d, **score
    })

score = run_experiment_augmentation(model,best_conf_resnet, None, cutmix_collate_fn=cutmix_collate_fn)
d = {
    "model": model,
    "augmentation": 'cutmix'}
    
results.append({
        **d, **score
    })

In [ ]:
# tutaj też sprawdzamy jak się modele zachowywały, wizualizacje i tym podobne etc.

## Few-shot learning

In [ ]:
#przyklad jak wybrać podzbior do fewshot learningu, wystarczy na wczytanym datasecie zastosować few_shot_subset

transformations = basic_transforms(augmentation_type=None, model_type='ResNet')

train_dataset = get_train_dataset(transform=transformations)
val_dataset, test_dataset = get_val_train_dataset(model_type='ResNet')

few_shot_train =  few_shot_subset(train_dataset)
few_shot_val =  few_shot_subset(val_dataset)
few_shot_test =  few_shot_subset(test_dataset)

### jak to zrobimy to w sumie już można zaczynać polerowanie wszystkiego i pisanie raportu.